[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/magnetotransport/magnetotransport_one_place.ipynb)

# Transport Coefficients in a Magnetic Field

This example computes the transport coefficients of a conductor in a magnetic field at one place in the conductor: its longitudinal resistance and Hall resistance under an electric current, and its Seebeck, Nernst and Righi–Leduc signals under a temperature gradient, as the carrier balance sweeps from electrons only through compensation to holes only. Each carrier holds an electric charge and an entropy, so an electrochemical potential fall pushes opposite charges opposite ways while a temperature fall pushes both species toward the cooler end, and collisions balance that push into a steady Drude drift. The magnetic field turns each drift sideways through its Hall angle, deflecting electrons and holes opposite ways. Far from the contacts of a long bar with insulated sides nothing varies from place to place, so one interior point determines all five bulk coefficients: current runs along the bar while neither charge nor entropy crosses its sides.

In planar spacetime joined with one degenerate entropy direction `s`, each carrier species is modelled as a parcel holding charge along time and entropy along `s`, `valence * (time + entropy_per_charge * entropy)`. A thermodynamic drive is a bivector touching space and the conserved quantities, and the flux it drives is an antibivector joining a carrier's spatial drift velocity with its parcel. Pairing the four drive planes against the linear response map `response(Drive)` and the imposed longitudinal thermal fall gathers the four interior bar conditions into one invertible map on drives. Reversing the magnetic field reverses each species' spatial rotor and maps `response` to its `.adjugate()`, while the [companion notebook](magnetotransport.ipynb) assembles the same response across a triangulated Hall bar and Corbino ring.

In the matrix notation of linear irreversible thermodynamics, the response is the $4\times 4$ kinetic matrix $J_i = \sum_j L_{ij}(B)\,X_j$ satisfying the Onsager–Casimir relation $L_{ij}(B) = L_{ji}(-B)$, and the longitudinal resistivity, Hall resistivity, Seebeck coefficient, Nernst coefficient, and Righi–Leduc ratio come from partial block inversions under the transverse open-circuit and adiabatic constraints $J_{e,y} = 0$ and $J_{s,y} = 0$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.thermodynamics.magnetotransport import render

# Planar spacetime joined with a degenerate direction s for entropy.
ga = Algebra("x+y+t-") * Algebra("s0")
space = Algebra("x+y+")
quantities = Algebra("t-s0")
mv = NumpyContext(ga).multivector

Drive = ga.gatype.bivector().derive.touching(space, quantities)               # Bivector+
x_direction = mv.x.dual()                                                     # [] Antivector-
y_direction = mv.y.dual()                                                     # [] Antivector-
time = -mv.t.dual()                                                           # [] Antivector-
entropy = mv.s.dual()                                                         # [] Antivector-

## Carrier drift and the Hall rotor

Each carrier species drifts where collisional drag, `velocity / mobility`, balances the combined push of the drive and the magnetic field. Commuting `magnetic + drive` with the moving parcel `parcel + valence * velocity` gives the electrostatic push, the thermal push, and the Lorentz turning at once. Because the commutator of a spatial magnetic plane with a spatial velocity reduces to their geometric product, solving `velocity == mobility * (push + valence * magnetic * velocity)` inverts `1 - valence * mobility * magnetic`, turning the rest push through the species' Hall angle and foreshortening it by the cosine.

In [ ]:
# Electrons and holes carry opposite charges, opposite entropy per unit charge, and unit mobility.
valences = np.array([-1, 1])                                                  # [2]
entropies_per_charge = np.array([-1.0, 1.0])                                  # [2]
mobilities = np.array([1.0, 1.0])                                             # [2]
strength = 2.0

In [ ]:
parcels = valences * (time + entropies_per_charge * entropy)                  # [2] Parcel-
magnetic = strength * mv.xy                                                   # [] Bivector+
# Commuting Drive with each rest parcel gives its push; inverting the scaled spatial rotor turns push into drift.
push = Drive.commutator(parcels)                                              # [2] Spatial- <- Drive+
hall_rotors = 1 - valences * mobilities * magnetic                            # [2] Rotor+
drifting = hall_rotors.inverse() * (mobilities * push)                        # [2] Spatial- <- Drive+

## Charge and entropy in one flux

Joining each species' drift velocity with its parcel via `drifting.regressive(parcels)` packs its charge current and entropy current into one antibivector flux: the planes `xt` and `yt` read charge carried along and across the bar, while `xs` and `ys` read entropy. Evaluating both single-species maps on a unit longitudinal electric drive shows how opposite Hall turns combine.

In [ ]:
carrying = drifting.regressive(parcels)                                       # [2] Flux- <- Drive+
field = mv.xt                                                                 # [] Drive+
render.turns_figure(field, carrying(field));

The crimson arrow marks the applied longitudinal electric drive. In the left charge readout, grey electron current and black hole current turn opposite ways so their compensated sum (violet) runs straight along the bar. In the right entropy readout, electrons carry entropy opposite to their negative charge, so at compensation the two species add into a transverse Ettingshausen entropy flux.

## Linear response across carrier balance

Sweeping the carrier balance from pure electrons (`-1`) through compensation (`0`) to pure holes (`+1`) at unit total carrier density weights the two carrier maps and adds lattice thermal conduction.

In [ ]:
conductivity = 1.5
n_balances = 101
balances = np.linspace(-1, 1, n_balances)                                     # [n_balances]
densities = np.stack([(1 - balances) / 2, (1 + balances) / 2], axis=-1)       # [n_balances, 2]

In [ ]:
conducting = conductivity * Drive.commutator(entropy).regressive(entropy)     # [] Flux- <- Drive+
response = (densities * carrying).sum(axis=-1) + conducting                   # [n_balances] Flux- <- Drive+

## Interior of an insulated bar

In the uniform interior of a long insulated bar, a drive satisfies four scalar readings corresponding to the four basis planes of `Drive`: the longitudinal charge current along `xt`, the longitudinal thermal fall along `xs`, and the vanishing transverse charge and entropy currents across `yt` and `ys`. Solving that combined map for unit longitudinal charge current gives the isothermal current-driven interior drive, and solving for unit longitudinal thermal fall gives the open-circuit heat-driven interior drive.

In [ ]:
conditions = (mv.xt * (mv.xt & response(Drive)) + mv.xs * (Drive & (x_direction & entropy))
              + mv.yt * (mv.yt & response(Drive)) + mv.ys * (mv.ys & response(Drive)))   # [n_balances] Drive+ <- Drive+
current_driven = conditions.solve(mv.xt)                                      # [n_balances] Drive+
heat_driven = conditions.solve(mv.xs)                                         # [n_balances] Drive+

resistance = current_driven & (x_direction & time)                            # [n_balances] Scalar+
hall_resistance = current_driven & (y_direction & time)                       # [n_balances] Scalar+
seebeck = -(heat_driven & (x_direction & time))                               # [n_balances] Scalar+
nernst = -(heat_driven & (y_direction & time))                                # [n_balances] Scalar+
righi_leduc = heat_driven & (y_direction & entropy)                           # [n_balances] Scalar+
render.readings_figure(balances, resistance, hall_resistance, seebeck, nernst, righi_leduc, render.balance_names);

At either single-carrier limit the transverse Hall field cancels Lorentz deflection, restoring unit longitudinal resistance and zero transverse Nernst or Righi–Leduc response. At compensation the Hall resistance passes through zero while longitudinal magnetoresistance and the transverse Nernst signal reach their peaks, and the Righi–Leduc temperature ratio changes sign across compensation.

In [ ]:
# checks
middle = n_balances // 2
assert ((hall_resistance[[0, -1]] - strength / valences).abs() < 1e-9).all()
assert ((resistance[[0, -1]] - 1.0).abs() < 1e-9).all()
assert hall_resistance[middle].abs() < 1e-9
assert nernst.abs().argmax() == middle
# Reversing the magnetic field maps the response to its adjugate, and dissipation remains strictly positive.
reversed_drifting = (1 + valences * mobilities * magnetic).inverse() * (mobilities * push)   # [2] Spatial- <- Drive+
reversed_response = (densities * reversed_drifting.regressive(parcels)).sum(axis=-1) + conducting   # [n_balances] Flux- <- Drive+
planes = stack((mv.xt, mv.yt, mv.xs, mv.ys))                                  # [4] Drive+
difference = reversed_response - response.adjugate()                          # [n_balances] Flux- <- Drive+
assert (planes[:, None, None] & difference(planes[None, :, None])).abs().max() < 1e-12
assert ((current_driven & response(current_driven)) > 0).all() and ((heat_driven & response(heat_driven)) > 0).all()